
# NUTS Level 0 Map — Debug Notebook

This notebook helps you **debug and render** the deck.gl map step-by-step using **JupyterDash** and **dash-deck**.

**What you'll do:**
1. Verify your environment and library versions
2. Load Parquet and filter `region_level == 2` (your "country" level)
3. Load the Level 0 GeoJSON and check feature counts
4. Confirm code matching between Parquet and GeoJSON
5. Render a **static-color** polygon layer + **smoke test** dots (Paris, Berlin, London)
6. Switch to **data-driven colors** by `emp` with a quantile legend


In [ ]:

# 1) Imports & versions
import os, json, sys, platform
from pathlib import Path
import numpy as np
import pandas as pd

from jupyter_dash import JupyterDash
from dash import html, dcc, Output, Input
import dash_bootstrap_components as dbc
import dash_deck

print("Python  :", sys.version.split()[0])
print("OS      :", platform.platform())
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("dash-deck present?:", hasattr(dash_deck, "DeckGL"))


In [ ]:

# 2) Paths & config — ADJUST THESE TWO PATHS IF NEEDED
PARQUET_PATH = Path(r"D:\Hitakshi\Dashboard\slm-dashboard\data\eu_labor_force_small.parquet")
GEOJSON_L0_PATH = Path(r"D:\Hitakshi\Dashboard\slm-dashboard\data\NUTS_RG_20M_2024_4326_LEVL_0.geojson")

# Optional: set MAPBOX_TOKEN env var in your shell if you want a basemap.
MAPBOX_TOKEN = os.environ.get("MAPBOX_TOKEN", "")
APP_TITLE = "EU Labor — Level 2 Debug"
THEME = dbc.themes.MINTY

def norm_code(x):
    if pd.isna(x):
        return None
    return str(x).strip().upper()


In [ ]:

# 3) Load Parquet & filter to region_level == 2
df_all = pd.read_parquet(PARQUET_PATH)
assert {"region_level", "region_code", "year", "emp"}.issubset(df_all.columns), "Parquet missing required columns."

df2 = df_all[df_all["region_level"] == 2].copy()
years = sorted(df2["year"].dropna().unique().tolist())
print("Rows @ level 2:", len(df2))
print("Years found   :", years[:10], "..." if len(years) > 10 else "")
assert len(years) > 0, "No years found for region_level==2."
display(df2.head(3))


In [ ]:

# 4) Load GeoJSON (Level 0 countries)
with open(GEOJSON_L0_PATH, "r", encoding="utf-8") as f:
    gj0 = json.load(f)

feats = gj0.get("features", [])
print("GeoJSON features:", len(feats))
if feats:
    print("First feature properties keys:", list((feats[0].get("properties") or {}).keys())[:20])


In [ ]:

# 5) Matching diagnostics — check that every parquet code exists in GeoJSON
parquet_codes = set(df2["region_code"].map(norm_code).dropna().unique())
geo_codes = set()
for feat in feats:
    props = feat.get("properties", {}) or {}
    nid = props.get("NUTS_ID") or props.get("nuts_id") or props.get("id") or props.get("ID")
    if nid is not None:
        geo_codes.add(norm_code(nid))

missing_in_geo = sorted(parquet_codes - geo_codes)
print("Unique parquet codes:", len(parquet_codes))
print("Unique geo NUTS_ID  :", len(geo_codes))
print("Parquet NOT in Geo  :", len(missing_in_geo))
if missing_in_geo[:10]:
    print("Examples:", missing_in_geo[:10])
assert len(missing_in_geo) == 0, "Some parquet codes are not present in the GeoJSON (Level 0)."


In [ ]:

# 6) Static-color map first (to confirm rendering), with smoke test dots
from copy import deepcopy

def initial_view_state():
    return dict(latitude=54.0, longitude=15.0, zoom=3.3, minZoom=2.0, maxZoom=10.5, pitch=0, bearing=0)

def enrich_static(geo):
    g = deepcopy(geo)
    for feat in g.get("features", []):
        props = feat.setdefault("properties", {})
        nid = norm_code(props.get("NUTS_ID") or props.get("nuts_id") or props.get("id") or props.get("ID"))
        name = props.get("NAME_LATN") or props.get("NAME_2021") or props.get("CNTR_NAME") or ""
        props["nuts_id"] = nid
        props["tooltip"] = f"{name} ({nid})"
    return g

gj_static = enrich_static(gj0)

layer_geo = {
    "@@type": "GeoJsonLayer",
    "id": "nuts-level0",
    "data": gj_static,
    "pickable": True,
    "stroked": True,
    "filled": True,
    "opacity": 0.9,
    "getFillColor": [8, 81, 156, 200],  # STATIC COLOR
    "getLineColor": [80, 80, 80],
    "lineWidthMinPixels": 0.75,
    "autoHighlight": True,
}

smoke_layer = {
    "@@type": "ScatterplotLayer",
    "id": "smoke",
    "data": [
        {"position": [2.3522, 48.8566]},
        {"position": [13.4050, 52.5200]},
        {"position": [-0.1276, 51.5074]},
    ],
    "getPosition": "@@=position",
    "getRadius": 40000,
    "radiusMinPixels": 4,
    "getFillColor": [255, 255, 255, 255],
    "pickable": False,
}

deck_kwargs = dict(
    initialViewState=initial_view_state(),
    layers=[layer_geo, smoke_layer],
    controller=True,
    tooltip={"text": "{properties.tooltip}"},
)

if MAPBOX_TOKEN:
    deck_kwargs["mapStyle"] = "mapbox://styles/mapbox/light-v11"

app = JupyterDash(__name__, external_stylesheets=[THEME])
app.layout = dbc.Container(
    [
        dbc.Navbar(dbc.Container([dbc.NavbarBrand("Static polygons + smoke dots")], fluid=True), color="primary", dark=True),
        dbc.Container([dash_deck.DeckGL(**deck_kwargs, mapboxKey=MAPBOX_TOKEN if MAPBOX_TOKEN else None, 
                                        style={"height": "70vh", "width": "100%", "borderRadius": "1rem", "overflow": "hidden"})], fluid=True),
    ], fluid=True
)

app.run_server(mode="inline", port=8051, debug=True)


In [ ]:

# 7) Data-driven fill by emp (quantiles). Choose a year from the list printed earlier.
from copy import deepcopy
sel_year = years[-1]  # choose latest; you can set e.g., sel_year = 2022

def emp_map_for_year(y):
    sub = df2[df2["year"] == y].copy().sort_values(["region_code"])
    return sub.groupby("region_code")["emp"].last().rename(index=norm_code).to_dict()

def colorize_by_emp(geo, emp_map):
    g = deepcopy(geo)
    vals = np.array([v for v in emp_map.values() if pd.notna(v)], dtype=float)
    if len(vals) == 0:
        return g, [], []
    num_classes = 7
    bins = np.quantile(vals, np.linspace(0, 1, num_classes + 1)).astype(float)
    for i in range(1, len(bins)):
        if bins[i] <= bins[i-1]:
            bins[i] = bins[i-1] + 1e-9
    palette = [
        [239, 243, 255],
        [198, 219, 239],
        [158, 202, 225],
        [107, 174, 214],
        [66, 146, 198],
        [33, 113, 181],
        [8, 81, 156],
    ]
    def color_for(x):
        if x is None or pd.isna(x):
            return [220, 220, 220, 180]
        idx = np.searchsorted(bins, float(x), side="right") - 1
        idx = max(0, min(idx, num_classes - 1))
        r, g_, b = palette[idx]
        return [int(r), int(g_), int(b), 200]

    for feat in g.get("features", []):
        props = feat.setdefault("properties", {})
        nid = norm_code(props.get("NUTS_ID") or props.get("nuts_id") or props.get("id") or props.get("ID"))
        name = props.get("NAME_LATN") or props.get("NAME_2021") or props.get("CNTR_NAME") or ""
        ev = emp_map.get(nid, None)
        props["nuts_id"] = nid
        props["emp"] = None if ev is None or pd.isna(ev) else float(ev)
        props["tooltip"] = f"{name} ({nid})\nemp: {props['emp'] if props['emp'] is not None else 'n/a'}"
        props["fillColor"] = color_for(props["emp"])

    legend_labels = [f"{bins[i]:,.0f} – {bins[i+1]:,.0f}" for i in range(len(bins)-1)]
    return g, palette, legend_labels

emp_map = emp_map_for_year(sel_year)
print(f"Selected year: {sel_year}; Emp values: {sum(pd.notna(list(emp_map.values())))} with data.")

gj_color, palette, labels = colorize_by_emp(gj0, emp_map)

layer_geo2 = {
    "@@type": "GeoJsonLayer",
    "id": "nuts-level0-colored",
    "data": gj_color,
    "pickable": True,
    "stroked": True,
    "filled": True,
    "opacity": 0.95,
    "getFillColor": "@@=properties.fillColor",
    "getLineColor": [80, 80, 80],
    "lineWidthMinPixels": 0.75,
    "autoHighlight": True,
}

deck_kwargs2 = dict(
    initialViewState=dict(latitude=54.0, longitude=15.0, zoom=3.3, minZoom=2.0, maxZoom=10.5, pitch=0, bearing=0),
    layers=[layer_geo2],
    controller=True,
    tooltip={"text": "{properties.tooltip}"},
)
if MAPBOX_TOKEN:
    deck_kwargs2["mapStyle"] = "mapbox://styles/mapbox/light-v11"

legend_children = []
for color, label in zip(palette, labels):
    r, g_, b = color
    legend_children.append(
        html.Div([
            html.Span(style={"display":"inline-block","width":"18px","height":"18px","borderRadius":"4px","marginRight":"8px",
                             "backgroundColor": f"rgb({r},{g_},{b})","border":"1px solid rgba(0,0,0,0.15)"}),
            html.Span(label)
        ], className="mb-1")
    )

app2 = JupyterDash(__name__ + "_emp", external_stylesheets=[THEME])
app2.layout = dbc.Container(
    [
        dbc.Navbar(dbc.Container([dbc.NavbarBrand(f"Emp choropleth — Year {sel_year}")], fluid=True), color="dark", dark=True),
        dbc.Row([
            dbc.Col(dash_deck.DeckGL(**deck_kwargs2, mapboxKey=MAPBOX_TOKEN if MAPBOX_TOKEN else None,
                                     style={"height":"70vh","width":"100%","borderRadius":"1rem","overflow":"hidden"}), md=9),
            dbc.Col(dbc.Card([dbc.CardHeader("Legend (emp)"), dbc.CardBody(legend_children)]), md=3),
        ], className="g-3"),
    ], fluid=True
)
app2.run_server(mode="inline", port=8052, debug=True)


In [ ]:

# 8) Inspect a sample feature's properties (to verify keys)
if feats:
    props = feats[0].get("properties", {})
    print("Sample feature properties:")
    for k in list(props.keys())[:30]:
        print(" ", k, ":", props[k])
